# DSC2026 — Prism-2B private inference on Kaggle 2×T4

Clean runner for the frozen Prism contract:

- base: `infgrad/Prism-Qwen3.5-Reranker-2B`
- LoRA alpha: **32**
- passage aggregation: **top2_max**
- max length: **1024**
- score: `logit(yes) - logit(no)`
- 2 independent workers: **1 Prism model per T4**

Attach a Kaggle Dataset containing exactly:

- `best_state.pt`
- `PRISM_PRIVATE_PASSAGES.pkl`

Set **Accelerator = GPU T4 x2** and **Internet = ON**.

The long inference cell streams both subprocesses live like a terminal. There is
no separate 30-second monitor cell anymore.


## 1. Install dependencies

In [ ]:
import sys, subprocess

def pip(*args, check=True):
    cmd = [sys.executable, "-m", "pip", *args]
    print("$", " ".join(cmd), flush=True)
    return subprocess.run(cmd, check=check)

# Kaggle currently ships an old torchao that conflicts with recent PEFT.
pip("uninstall", "-y", "torchao", check=False)

# Pin the versions that recognize qwen3_5_text and work with this notebook.
pip(
    "install", "-q", "--no-cache-dir", "-U",
    "transformers==5.17.0",
    "peft==0.21.0",
    "accelerate",
    "safetensors",
    "sentencepiece",
)

# Optional Qwen3.5 fast kernels. Failure is non-fatal: Transformers can fall
# back to the slower reference PyTorch implementations.
optional = [
    ["install", "-q", "--no-build-isolation", "causal-conv1d"],
    ["install", "-q", "flash-linear-attention==0.5.2"],
]
for args in optional:
    try:
        pip(*args)
    except subprocess.CalledProcessError as e:
        print("WARNING: optional kernel install failed:", args[-1])
        print("Inference can still run with the slower fallback implementation.")

# Verify from a FRESH subprocess — this is exactly what the GPU workers will see.
verify = r"""
import transformers, peft
from transformers import Qwen3_5TextConfig, Qwen3_5ForCausalLM
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("Qwen3.5 architecture import: PASS")
try:
    import causal_conv1d
    print("causal_conv1d: PASS")
except Exception as e:
    print("causal_conv1d: unavailable:", type(e).__name__, e)
try:
    import fla
    print("flash-linear-attention/fla: PASS")
except Exception as e:
    print("flash-linear-attention/fla: unavailable:", type(e).__name__, e)
"""
subprocess.check_call([sys.executable, "-c", verify])


## 2. Verify GPUs and locate inputs

In [ ]:
import os, sys, subprocess
from pathlib import Path
import torch

print("Python:", sys.executable)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {p.name} | {p.total_memory/2**30:.1f} GiB")

assert torch.cuda.device_count() >= 2, (
    "This notebook expects Kaggle accelerator GPU T4 x2."
)

INPUT = Path("/kaggle/input")
checkpoint_hits = list(INPUT.rglob("best_state.pt"))
workload_hits = list(INPUT.rglob("PRISM_PRIVATE_PASSAGES.pkl"))

print("checkpoint hits:", checkpoint_hits)
print("workload hits:", workload_hits)

assert len(checkpoint_hits) == 1, (
    f"Expected exactly one best_state.pt, got {len(checkpoint_hits)}"
)
assert len(workload_hits) == 1, (
    f"Expected exactly one PRISM_PRIVATE_PASSAGES.pkl, got {len(workload_hits)}"
)

CHECKPOINT = checkpoint_hits[0]
WORKLOAD = workload_hits[0]

OUT = Path("/kaggle/working/prism_private")
OUT.mkdir(parents=True, exist_ok=True)

print("CHECKPOINT:", CHECKPOINT)
print("WORKLOAD:", WORKLOAD)
print("OUT:", OUT)


## 3. Write the GPU worker

In [ ]:
from pathlib import Path

WORKER = 'import argparse, os, pickle, time\nfrom pathlib import Path\n\n# Keep subprocess logs clean before importing HF/torch.\nos.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")\nos.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")\nos.environ.setdefault("TRANSFORMERS_NO_ADVISORY_WARNINGS", "1")\nos.environ.setdefault("TOKENIZERS_PARALLELISM", "false")\n\nimport numpy as np\nimport torch\n\nBASE_MODEL = "infgrad/Prism-Qwen3.5-Reranker-2B"\n\nSYSTEM_PROMPT = (\n    "Judge whether the Document meets the requirements based on "\n    "the Query and the Instruct provided. "\n)\n\nINSTRUCTION = (\n    \'Judge if the document is relevant to the query. Reply "yes" or "no".\\n\'\n    \'On "yes", also emit:\\n\'\n    "<contribution>One sentence covering every core point the document "\n    "contributes to the query, without elaboration.</contribution>\\n"\n    "<evidence>Self-contained rewrite of the query-relevant content. Rules:\\n"\n    "- Faithful: rephrase only; add or infer nothing.\\n"\n    "- Self-contained: evidence alone must fully answer the query.\\n"\n    "- Concise: drop query-irrelevant background.\\n"\n    "- Verbatim (no translation): proper nouns, terms, abbreviations, "\n    "numbers, dates, code, URLs.\\n"\n    "- Output language: multilingual doc → query\'s language; else doc\'s language."\n    "</evidence>"\n)\n\nTEMPLATE = (\n    "<|im_start|>system\\n{system}<|im_end|>\\n"\n    "<|im_start|>user\\n"\n    "<Instruct>: {instruction}\\n"\n    "<Query>: {query}\\n"\n    "<Document>: {doc}<|im_end|>\\n"\n    "<|im_start|>assistant\\n<think>\\n\\n</think>\\n\\n"\n)\n\ndef mkprompt(q, d):\n    return TEMPLATE.format(\n        system=SYSTEM_PROMPT,\n        instruction=INSTRUCTION,\n        query=q,\n        doc=d,\n    )\n\ndef atomic_pickle(path, obj):\n    path = Path(path)\n    tmp = path.with_suffix(path.suffix + ".tmp")\n    tmp.write_bytes(pickle.dumps(obj, protocol=5))\n    os.replace(tmp, path)\n\ndef infer_contract(state):\n    a_keys = [k for k in state if ".lora_A." in k]\n    if not a_keys:\n        raise RuntimeError("No LoRA A tensors found in checkpoint")\n    ranks = {int(state[k].shape[0]) for k in a_keys}\n    if len(ranks) != 1:\n        raise RuntimeError(f"Mixed LoRA ranks: {ranks}")\n    rank = next(iter(ranks))\n    targets = sorted({\n        k.split(".lora_A.")[0].split(".")[-1]\n        for k in a_keys\n    })\n    return rank, targets\n\ndef load_model(checkpoint, gpu):\n    from transformers import AutoModelForCausalLM, AutoTokenizer\n    from transformers.utils import logging as hf_logging\n    from peft import LoraConfig, TaskType, get_peft_model\n\n    hf_logging.disable_progress_bar()\n\n    torch.cuda.set_device(gpu)\n    device = f"cuda:{gpu}"\n\n    print(f"loading checkpoint metadata: {checkpoint}", flush=True)\n    obj = torch.load(checkpoint, map_location="cpu", weights_only=True)\n    state = obj.get("state_dict", obj)\n    rank, targets = infer_contract(state)\n    print(\n        f"LoRA: rank={rank} alpha=32 tensors={len(state)} "\n        f"targets={\',\'.join(targets)}",\n        flush=True,\n    )\n\n    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)\n    tokenizer.padding_side = "left"\n    if tokenizer.pad_token_id is None:\n        tokenizer.pad_token = tokenizer.eos_token\n\n    print(f"loading base model: {BASE_MODEL}", flush=True)\n    base = AutoModelForCausalLM.from_pretrained(\n        BASE_MODEL,\n        dtype=torch.float16,\n        low_cpu_mem_usage=True,\n        attn_implementation="sdpa",\n    )\n\n    cfg = LoraConfig(\n        r=rank,\n        lora_alpha=32,\n        target_modules=targets,\n        lora_dropout=0.0,\n        bias="none",\n        task_type=TaskType.CAUSAL_LM,\n    )\n    model = get_peft_model(base, cfg)\n    incompat = model.load_state_dict(state, strict=False)\n    if incompat.unexpected_keys:\n        raise RuntimeError(\n            f"Unexpected checkpoint keys: {incompat.unexpected_keys[:10]}"\n        )\n\n    model.eval().to(device)\n\n    yes_ids = tokenizer.encode("yes", add_special_tokens=False)\n    no_ids = tokenizer.encode("no", add_special_tokens=False)\n    if len(yes_ids) != 1 or len(no_ids) != 1:\n        raise RuntimeError(\n            f"Unexpected yes/no tokenization: yes={yes_ids}, no={no_ids}"\n        )\n\n    alloc = torch.cuda.memory_allocated(gpu) / 2**30\n    reserved = torch.cuda.memory_reserved(gpu) / 2**30\n    print(\n        f"model ready | device={device} | "\n        f"VRAM allocated={alloc:.2f} GiB reserved={reserved:.2f} GiB",\n        flush=True,\n    )\n    return model, tokenizer, yes_ids[0], no_ids[0], device\n\n@torch.inference_mode()\ndef score_batch(model, tokenizer, yes_id, no_id, device, texts, max_length):\n    enc = tokenizer(\n        texts,\n        padding=True,\n        truncation=True,\n        max_length=max_length,\n        return_tensors="pt",\n        add_special_tokens=False,\n    )\n    enc = {\n        k: v.to(device, non_blocking=True)\n        for k, v in enc.items()\n    }\n\n    # PeftModel\'s base modules still contain the injected LoRA layers.\n    base = model.get_base_model()\n    with torch.autocast(device_type="cuda", dtype=torch.float16):\n        out = base.model(\n            input_ids=enc["input_ids"],\n            attention_mask=enc.get("attention_mask"),\n            use_cache=False,\n            return_dict=True,\n        )\n        hidden = out.last_hidden_state[:, -1, :]\n        logits = base.lm_head(hidden).float()\n        scores = logits[:, yes_id] - logits[:, no_id]\n\n    return scores.detach().cpu().tolist()\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--gpu", type=int, required=True)\n    ap.add_argument("--shard", type=int, required=True)\n    ap.add_argument("--num-shards", type=int, required=True)\n    ap.add_argument("--workload", required=True)\n    ap.add_argument("--checkpoint", required=True)\n    ap.add_argument("--output", required=True)\n    ap.add_argument("--batch-size", type=int, default=32)\n    ap.add_argument("--max-length", type=int, default=1024)\n    ap.add_argument("--save-every", type=int, default=25)\n    ap.add_argument("--log-every", type=int, default=5)\n    args = ap.parse_args()\n\n    print(\n        f"worker start | gpu={args.gpu} shard={args.shard}/{args.num_shards} "\n        f"batch={args.batch_size} max_length={args.max_length}",\n        flush=True,\n    )\n\n    model, tokenizer, yes_id, no_id, device = load_model(\n        args.checkpoint, args.gpu\n    )\n\n    workload_obj = pickle.loads(Path(args.workload).read_bytes())\n    queries = workload_obj["queries"]\n    all_qids = sorted(queries)\n    qids = [\n        q for i, q in enumerate(all_qids)\n        if i % args.num_shards == args.shard\n    ]\n\n    output_path = Path(args.output)\n    saved = (\n        pickle.loads(output_path.read_bytes())\n        if output_path.is_file()\n        else {}\n    )\n\n    # Only trust complete qids on resume.\n    remain = [\n        q for q in qids\n        if q not in saved\n        or len(saved[q]) != len(queries[q]["docs"])\n    ]\n\n    print(\n        f"workload ready | shard_queries={len(qids)} "\n        f"cached={len(qids)-len(remain)} remaining={len(remain)}",\n        flush=True,\n    )\n\n    if not remain:\n        print("nothing to do: shard already complete", flush=True)\n        return\n\n    bs = args.batch_size\n    start = time.perf_counter()\n\n    for qi, qid in enumerate(remain, 1):\n        row = queries[qid]\n        owners = []\n        prompts = []\n\n        for doc_id, passages in row["docs"].items():\n            for passage in passages:\n                owners.append(str(doc_id))\n                prompts.append(mkprompt(row["question"], passage))\n\n        values = []\n        pos = 0\n        while pos < len(prompts):\n            chunk = prompts[pos:pos + bs]\n            try:\n                values.extend(\n                    score_batch(\n                        model,\n                        tokenizer,\n                        yes_id,\n                        no_id,\n                        device,\n                        chunk,\n                        args.max_length,\n                    )\n                )\n                pos += len(chunk)\n            except torch.cuda.OutOfMemoryError:\n                torch.cuda.empty_cache()\n                if bs <= 1:\n                    raise\n                old_bs = bs\n                bs = max(1, bs // 2)\n                print(\n                    f"CUDA OOM at q={qid}: batch {old_bs} -> {bs}; retrying",\n                    flush=True,\n                )\n\n        scores = {\n            str(doc_id): -1e30\n            for doc_id in row["docs"]\n        }\n        for doc_id, score in zip(owners, values):\n            if not np.isfinite(score):\n                raise RuntimeError(\n                    f"non-finite score q={qid} doc={doc_id}: {score}"\n                )\n            scores[doc_id] = max(scores[doc_id], float(score))\n\n        saved[qid] = scores\n\n        if qi % args.save_every == 0:\n            atomic_pickle(output_path, saved)\n\n        if qi == 1 or qi % args.log_every == 0 or qi == len(remain):\n            elapsed = time.perf_counter() - start\n            sec_per_q = elapsed / qi\n            eta_min = sec_per_q * (len(remain) - qi) / 60\n            done_total = len(qids) - len(remain) + qi\n            print(\n                f"progress {done_total}/{len(qids)} | "\n                f"new {qi}/{len(remain)} | "\n                f"{sec_per_q:.2f}s/q | ETA {eta_min:.1f}m | "\n                f"batch={bs} | prompts_last_q={len(prompts)}",\n                flush=True,\n            )\n\n    atomic_pickle(output_path, saved)\n    elapsed = time.perf_counter() - start\n    print(\n        f"COMPLETE | saved={len(saved)}/{len(qids)} "\n        f"| elapsed={elapsed/60:.1f}m | output={output_path}",\n        flush=True,\n    )\n\nif __name__ == "__main__":\n    main()\n'

WORKER_PATH = Path('/kaggle/working/prism_worker.py')
WORKER_PATH.write_text(WORKER, encoding='utf-8')
print('worker:', WORKER_PATH)
print('bytes:', WORKER_PATH.stat().st_size)


## 4. Run both T4s with live terminal output

This single cell:

1. terminates stale `prism_worker.py` processes from an older attempt;
2. launches one worker on GPU 0 and one on GPU 1;
3. streams each subprocess line immediately with `[GPU0]` / `[GPU1]`;
4. waits for both workers to finish;
5. fails if either worker exits non-zero.

`batch=32` is only the starting value. On CUDA OOM, that worker automatically
falls back `32 → 16 → 8 → 4 → 2 → 1` and retries the same chunk.

Partial shard caches are preserved and resumed.


In [ ]:
import os, sys, time, queue, threading, subprocess

# Kill only stale workers from an earlier notebook attempt.
subprocess.run(
    ["pkill", "-f", "/kaggle/working/prism_worker.py"],
    check=False,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
time.sleep(1)

env = os.environ.copy()
env["TOKENIZERS_PARALLELISM"] = "false"
env["PYTHONUNBUFFERED"] = "1"
env["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
env["TRANSFORMERS_VERBOSITY"] = "error"
env["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
env.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

START_BATCH = 32
SAVE_EVERY = 25
LOG_EVERY = 5

procs = []
events = queue.Queue()

def pump(gpu, proc):
    try:
        for line in proc.stdout:
            events.put((gpu, line.rstrip("\n")))
    finally:
        events.put((gpu, None))

for gpu in (0, 1):
    cmd = [
        sys.executable,
        str(WORKER_PATH),
        "--gpu", str(gpu),
        "--shard", str(gpu),
        "--num-shards", "2",
        "--workload", str(WORKLOAD),
        "--checkpoint", str(CHECKPOINT),
        "--output", str(OUT / f"prism_scores_shard{gpu}.pkl"),
        "--batch-size", str(START_BATCH),
        "--max-length", "1024",
        "--save-every", str(SAVE_EVERY),
        "--log-every", str(LOG_EVERY),
    ]

    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )
    procs.append(proc)

    t = threading.Thread(
        target=pump,
        args=(gpu, proc),
        daemon=True,
    )
    t.start()
    print(f"[MAIN] launched GPU{gpu} pid={proc.pid}", flush=True)

finished_streams = 0

try:
    while finished_streams < len(procs):
        try:
            gpu, line = events.get(timeout=1.0)
        except queue.Empty:
            continue

        if line is None:
            finished_streams += 1
        else:
            print(f"[GPU{gpu}] {line}", flush=True)

except KeyboardInterrupt:
    print("\n[MAIN] KeyboardInterrupt — terminating workers...", flush=True)
    for proc in procs:
        if proc.poll() is None:
            proc.terminate()
    raise

codes = [proc.wait() for proc in procs]
print("[MAIN] return codes:", codes, flush=True)

if codes != [0, 0]:
    raise RuntimeError(
        f"At least one Prism worker failed: return codes={codes}"
    )

print("[MAIN] BOTH SHARDS COMPLETE", flush=True)


## 5. Merge, validate, and create the final score cache

In [ ]:
import pickle, hashlib, json
from pathlib import Path

workload_obj = pickle.loads(WORKLOAD.read_bytes())
queries = workload_obj["queries"]

merged = {}
for gpu in (0, 1):
    shard_path = OUT / f"prism_scores_shard{gpu}.pkl"
    assert shard_path.is_file(), f"Missing shard: {shard_path}"

    part = pickle.loads(shard_path.read_bytes())
    overlap = set(merged) & set(part)
    assert not overlap, f"Shard qid overlap: {list(overlap)[:10]}"
    merged.update(part)

assert len(merged) == 2080, (
    f"Expected 2080 qids, got {len(merged)}"
)

pairs = 0
missing = []

for qid, row in queries.items():
    docs = row["docs"]
    pairs += len(docs)

    if qid not in merged:
        missing.append((qid, None))
        continue

    for doc_id in docs:
        if doc_id not in merged[qid]:
            missing.append((qid, doc_id))
            if len(missing) >= 20:
                break

assert not missing, f"Missing Prism scores: {missing[:20]}"

final_path = Path("/kaggle/working/prism_private_scores.pkl")
final_path.write_bytes(
    pickle.dumps(merged, protocol=5)
)

sha256 = hashlib.sha256(final_path.read_bytes()).hexdigest()
values = [
    float(score)
    for row in merged.values()
    for score in row.values()
]

report = {
    "schema": "manual.kaggle_prism_private_2xt4.v2",
    "queries": len(merged),
    "pairs": pairs,
    "score_min": min(values),
    "score_max": max(values),
    "sha256": sha256,
    "base": "infgrad/Prism-Qwen3.5-Reranker-2B",
    "alpha": 32,
    "mode": "top2_max",
    "max_length": 1024,
}

report_path = Path("/kaggle/working/PRISM_KAGGLE_REPORT.json")
report_path.write_text(
    json.dumps(report, indent=2),
    encoding="utf-8",
)

print(json.dumps(report, indent=2))
print()
print("FINAL SCORE CACHE:", final_path)
print("REPORT:", report_path)
